<a href="https://colab.research.google.com/github/Ajaymani9345/Movie-Recommendation-System/blob/main/movie.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
uploaded = files.upload()

Saving Dataset.xlsx to Dataset.xlsx


In [2]:
!pip install geopy scikit-learn folium openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.4/125.4 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.4/113.4 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.9/250.9 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.1/94.1 kB 9.5 MB/s eta 0:00:00


In [4]:

import pandas as pd
import numpy as np
from geopy.geocoders import Nominatim
from geopy.distance import geodesic
from sklearn.cluster import KMeans
import folium
import time
df = pd.read_excel('Dataset.xlsx')
try:
    df['Area (ha)'] = pd.to_numeric(df['Area (ha)'], errors='raise')
except ValueError:
    df['Area (ha)'] = pd.to_numeric(df['Area (ha)'].astype(str).str.replace(',', ''), errors='coerce')

df = df.dropna(subset=['Area (ha)'])
geolocator = Nominatim(user_agent="tn_agri_office_v3")
district_coords = {
    'Salem': (11.6500, 78.1333),
    'Theni': (10.0104, 77.4768),
    'Tiruchirappalli': (10.7833, 78.6833),
    'Dharmapuri': (12.1065, 78.1361),
    'Kallakurichi': (11.7380, 78.9620),
    'Tiruppur': (11.1107, 77.3480),
    'Thanjavur': (10.7870, 79.1378),
    'Kanniyakumari': (8.0883, 77.5385),
    'Tiruvannamalai': (12.2286, 79.0665),
    'Cuddalore': (11.7200, 79.7700),
    'Villupuram': (11.9400, 79.5000),
    'Namakkal': (11.2186, 78.1682),
    'Dindigul': (10.3650, 77.9800),
    'Erode': (11.3428, 77.7274),
    'Thoothukudi': (8.7642, 78.1348),
    'Nilgiris': (11.4000, 76.7000),
    'Coimbatore': (11.0168, 76.9558)
}
def get_coordinates(district, taluk):
    try:
        location = geolocator.geocode(
            f"{taluk}, {district}, Tamil Nadu, India",
            timeout=10
        )
        if location:
            return (location.latitude, location.longitude)

        return district_coords.get(district, (None, None))

    except Exception as e:
        print(f"Geocoding failed for {taluk}, {district}: {str(e)}")
        return district_coords.get(district, (None, None))

    finally:
        time.sleep(1)
print("Geocoding locations...")
df['Coordinates'] = df.apply(lambda x: get_coordinates(x['District'], x['Taluk']), axis=1)
df = df.dropna(subset=['Coordinates'])
df[['Latitude', 'Longitude']] = pd.DataFrame(df['Coordinates'].tolist(), index=df.index)
def calculate_weighted_centroid(crop_df):
    valid_rows = crop_df.dropna(subset=['Latitude', 'Longitude', 'Area (ha)'])

    if len(valid_rows) == 0:
        return None
    total_area = valid_rows['Area (ha)'].sum()
    if total_area == 0:
        return None
    weighted_lat = np.sum(valid_rows['Latitude'] * valid_rows['Area (ha)']) / total_area
    weighted_lon = np.sum(valid_rows['Longitude'] * valid_rows['Area (ha)']) / total_area
    def distance(row):
        return geodesic((weighted_lat, weighted_lon), (row['Latitude'], row['Longitude'])).km
    valid_rows = valid_rows.copy()
    valid_rows.loc[:, 'distance'] = valid_rows.apply(distance, axis=1)
    nearest = valid_rows.loc[valid_rows['distance'].idxmin()]
    return {
        'crop': nearest['Crop'],
        'centroid_lat': weighted_lat,
        'centroid_lon': weighted_lon,
        'recommended_district': nearest['District'],
        'recommended_taluk': nearest['Taluk'],
        'total_area': total_area,
        'num_locations': len(valid_rows)
    }
crop_centroids = {}
for crop in df['Crop'].unique():
    centroid = calculate_weighted_centroid(df[df['Crop'] == crop])
    if centroid:
        crop_centroids[crop] = centroid
def optimize_agri_offices(crop_name, max_offices=3):
    if crop_name not in crop_centroids:
        return {"error": f"Crop '{crop_name}' not found"}

    crop_df = df[df['Crop'] == crop_name].dropna(subset=['Latitude', 'Longitude', 'Area (ha)'])

    if len(crop_df) <= 2:
        best = crop_df.sort_values('Area (ha)', ascending=False).iloc[0]
        return {
            "recommendations": [{
                "district": best['District'],
                "taluk": best['Taluk'],
                "latitude": best['Latitude'],
                "longitude": best['Longitude'],
                "area_covered": float(best['Area (ha)']),
                "coverage_percentage": 100.0
            }]
        }

    X = crop_df[['Latitude', 'Longitude']].values
    weights = crop_df['Area (ha)'].values

    n_clusters = min(max_offices, max(1, len(crop_df) // 5))

    weighted_samples = np.repeat(X, weights.astype(int), axis=0)

    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    kmeans.fit(weighted_samples)

    recommendations = []

    for center in kmeans.cluster_centers_:
        distances = np.array([geodesic(center, x).km for x in X])
        cluster_mask = distances < np.percentile(distances, 75)

        if not cluster_mask.any():
            continue

        best = crop_df[cluster_mask].sort_values('Area (ha)', ascending=False).iloc[0]
        cluster_area = crop_df[cluster_mask]['Area (ha)'].sum()

        recommendations.append({
            "district": best['District'],
            "taluk": best['Taluk'],
            "latitude": float(center[0]),
            "longitude": float(center[1]),
            "area_covered": float(cluster_area),
            "coverage_percentage": float(round(100 * cluster_area / crop_df['Area (ha)'].sum(), 2))
        })
    unique = {}
    for rec in recommendations:
        key = (rec['district'], rec['taluk'])
        if key not in unique:
            unique[key] = rec

    return {"recommendations": list(unique.values())}
print("Banana offices:")
print(optimize_agri_offices("Banana"))

Geocoding locations...
Banana offices:
{'recommendations': [{'district': 'Erode', 'taluk': 'Modakurichi', 'latitude': 11.025802837231126, 'longitude': 77.1310044227032, 'area_covered': 39521.0, 'coverage_percentage': 78.15}, {'district': 'Erode', 'taluk': 'Kodumudi', 'latitude': 9.318121483745127, 'longitude': 77.75421302860858, 'area_covered': 32694.0, 'coverage_percentage': 64.65}]}
